# Training PPO Tiny Tackers
Run on Google Colab for superior training efficiency
I recommend the T4 GPU

- - - - - - - - - - - - - - - - - - - - - - - -
## Train on gym_sail_steps SailboatReachRounding Environment

This notebook goes through the learning tasks in the same order required for the boat to complete a Triangle Racecourse. Each leg of that race demands that the PPO agent learn new skills.

The first learning task is to complete the environment built by Gabo Tor as this environment is the inspiration for the project. Subsequent learning tasks are all trained in environments created by yours truly. They are as follows:

    1. Start near the leeward buoy and reach the target point at the windward buoy
    2. Start near the windward buoy and reach the target point at the reach buoy
    3. Start near the reach buoy and reach the target point at the leeward buoy
    4. Start near the windward buoy and round the reach buoy finishing at the leeward buoy, all while chasing target points that are designed to encourage jibing
    5. Start near the windward buoy and reach the target point near the leeward buoy
    6. Hit every target point in a triangle race course by rounding all the buoys
- - - - - - - - - - - - - - - - - - - - - - - -



#####
#####
# *"Sailing is the art of going nowhere slowly, at great expense"*
*- The sailing community*


#####
#####
#####
#####

- - - - - - - - - - - - - - - - - - - -
## Training a PPO Baseline Model on Gabo-Tor Environment
- Learning Outcome: The Tiny Tacker must learn how to sail into the wind on a close haul while also learning to tack (turn through the wind without getting stuck in irons)
  1. Reach the Windward Buoy to Finish
- - - - - - - - - - - - - - - - - - - -

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set directory to tiny_tackers
%cd "/content/drive/My Drive/tiny_tackers"

In [ ]:
# Install and import dependencies
!pip install stable-baselines3 gymnasium pygame

import os
import sys
import time
import pandas as pd
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

from gymnasium.wrappers import RecordVideo

In [ ]:
# Set path to retrieving the base environment
BASE_ENV_PATH = "/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor"
sys.path.append(BASE_ENV_PATH)

In [ ]:
# Import environment and ensure it's comming from the correct path. Ending in tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor/gym_sailing/__init__.py
import gym_sailing
print(gym_sailing.__file__)

In [ ]:
# Specifify that the environment is going to be one where continuous actions are possible
ENV_ID = "Sailboat-v0"

In [ ]:
# Create directory for storing your model, videos, and metrics
models_dir = "models/ppo/base"
videos_dir = "videos/ppo/base"
metrics_dir = "metrics"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(videos_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

In [ ]:
# Make and define training and evaluation environments (render mode is none because vizualizing significantly slows down training and evaluation)
def make_env(render_mode=None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    env = Monitor(env)
    return env

train_env = make_env()
eval_env = make_env()

In [ ]:
# Improve efficiency with parallel environments
from stable_baselines3.common.env_util import make_vec_env
train_env = make_vec_env(lambda: make_env(), n_envs=4)

In [ ]:
# Train PPO for 1_000_000 timesteps and save the training time in seconds
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    gamma=0.99,
    verbose=1,
)

start = time.time()
model.learn(
    total_timesteps=1_000_000,
    progress_bar=True,
)

training_time_seconds = time.time() - start

In [ ]:
# Evaluate and capture average episode length and rewards over 20 episodes and view the results
episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True,
    return_episode_rewards=True,
)

mean_reward = sum(episode_rewards) / len(episode_rewards)
std_reward = pd.Series(episode_rewards).std()

mean_episode_timesteps = sum(episode_lengths) / len(episode_lengths)
std_episode_timesteps = pd.Series(episode_lengths).std()

print("Mean reward:", mean_reward)
print("Std reward:", std_reward)
print("Mean episode timesteps:", mean_episode_timesteps)
print("Std episode timesteps:", std_episode_timesteps)
print("Training time seconds:", training_time_seconds)

In [ ]:
# Save the base model if you are satisfied
model_path = f"{models_dir}/ppo_base_1M"
model.save(model_path)

print(f"Saved model to: {model_path}.zip")

In [ ]:
# Save metrics if you are satisfied.
metrics = pd.DataFrame([{
    "env_id": ENV_ID,
    "model": "PPO",
    "training_timesteps": 1_000_000,
    "n_eval_episodes": 20,
    "mean_reward": mean_reward,
    "std_reward": std_reward,
    "mean_episode_timesteps": mean_episode_timesteps,
    "std_episode_timesteps": std_episode_timesteps,
    "training_time_seconds": training_time_seconds,
}])

metrics_path = f"{metrics_dir}/ppo_base_metrics.csv"

if os.path.exists(metrics_path):
    old = pd.read_csv(metrics_path)
    metrics = pd.concat([old, metrics], ignore_index=True)

metrics.to_csv(metrics_path, index=False)
metrics

In [ ]:
# Record video an evaluation episode of the trained model and save it in the videos directory with a name prefix of ppo_base_trained. The video should be saved as a mp4 file and should be named ppo_base_trained_episode_0.mp4.
video_env = gym.make(ENV_ID, render_mode="rgb_array")
video_env = RecordVideo(
    video_env,
    video_folder=videos_dir,
    name_prefix="ppo_base_trained",
    episode_trigger=lambda episode_id: episode_id == 0,
)

obs, info = video_env.reset()
done = False
truncated = False

while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = video_env.step(action)

import builtins # imported to fix bug with recording video
builtins.quit = lambda *args, **kwargs: None # imported to fix bug with recording video
video_env.close()
